In [ ]:
# ============================================================
# INSIDER THREAT DETECTION
# ELASTIC NET LOGISTIC REGRESSION - PREDICTION DEMO
# ============================================================

import pandas as pd
import numpy as np

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. UPLOAD DATASET
# ============================================================

print("=" * 70)
print("INSIDER THREAT DETECTION")
print("=" * 70)

uploaded = files.upload()

filename = next(iter(uploaded))

df = pd.read_csv(filename)

print("\nDataset loaded successfully!")
print("Rows    :", df.shape[0])
print("Columns :", df.shape[1])


# ============================================================
# 2. CHECK TARGET
# ============================================================

if "RiskLabel" not in df.columns:
    raise ValueError("RiskLabel column not found.")

print("\nRisk labels:")
print(df["RiskLabel"].value_counts())


# ============================================================
# 3. CONVERT RISK LABEL TO BINARY TARGET
#
# Low              -> Normal
# Medium + High    -> Potential Insider
# ============================================================

label_map = {
    "low": 0,
    "medium": 1,
    "high": 1
}

df["Target"] = (
    df["RiskLabel"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(label_map)
)

if df["Target"].isna().any():
    raise ValueError(
        "Unexpected values found in RiskLabel:\n"
        + str(df.loc[df["Target"].isna(), "RiskLabel"].unique())
    )


# ============================================================
# 4. SELECT ACTIVITY FEATURES
# ============================================================

flag_features = [
    col for col in df.columns
    if col.endswith("_flag")
]

if len(flag_features) == 0:
    raise ValueError("No activity *_flag columns found.")

print("\nActivity features used:")
for feature in flag_features:
    print("-", feature)

print("\nTotal activity features:", len(flag_features))


# ============================================================
# 5. PREPARE FEATURES
# ============================================================

X = df[flag_features].copy()
y = df["Target"].copy()

for col in X.columns:
    X[col] = pd.to_numeric(
        X[col],
        errors="coerce"
    )

    X[col] = X[col].fillna(
        X[col].median()
    )


# ============================================================
# 6. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 7. ELASTIC NET LOGISTIC REGRESSION
# ============================================================

model = Pipeline([

    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            l1_ratio=0.5,
            C=1.0,
            max_iter=5000,
            random_state=42
        )
    )
])


# ============================================================
# 8. TRAIN
# ============================================================

print("\nTraining model...")

model.fit(
    X_train,
    y_train
)

print("Model trained successfully!")


# ============================================================
# 9. MODEL EVALUATION
# ============================================================

y_pred = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\n" + "=" * 70)
print("MODEL EVALUATION")
print("=" * 70)

print(
    f"\nAccuracy: {accuracy * 100:.2f}%"
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Normal",
            "Potential Insider"
        ]
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


# ============================================================
# 10. USER INPUT
# ============================================================

print("\n" + "=" * 70)
print("ENTER NEW USER ACTIVITY")
print("=" * 70)

print(
    "\nEnter 0 = No"
)

print(
    "Enter 1 = Yes"
)

user_data = {}

for feature in flag_features:

    while True:

        value = input(
            f"\n{feature}: "
        ).strip()

        if value in ["0", "1"]:

            user_data[feature] = int(value)

            break

        print("Please enter only 0 or 1.")


# ============================================================
# 11. CREATE INPUT DATAFRAME
# ============================================================

new_user = pd.DataFrame(
    [user_data],
    columns=flag_features
)


# ============================================================
# 12. PREDICT
# ============================================================

prediction = model.predict(
    new_user
)[0]

probability = model.predict_proba(
    new_user
)[0]


# ============================================================
# 13. FINAL RESULT
# ============================================================

print("\n" + "=" * 70)
print("FINAL PREDICTION")
print("=" * 70)

if prediction == 0:

    print("\nPrediction: NORMAL")

else:

    print("\nPrediction: POTENTIAL INSIDER")


print(
    f"\nNormal probability: "
    f"{probability[0] * 100:.2f}%"
)

print(
    f"Potential Insider probability: "
    f"{probability[1] * 100:.2f}%"
)


print("\n" + "=" * 70)

if prediction == 0:

    print(
        "The entered activity pattern is classified "
        "as NORMAL by the model."
    )

else:

    print(
        "The entered activity pattern is classified "
        "as POTENTIALLY SUSPICIOUS by the model."
    )

print(
    "\nNote: This prediction identifies a behavioral "
    "pattern and does not prove malicious intent."
)

print("=" * 70)

INSIDER THREAT DETECTION


Saving Insiders Abnormal Activities.csv to Insiders Abnormal Activities.csv

Dataset loaded successfully!
Rows    : 23500
Columns : 82

Risk labels:
RiskLabel
Medium    10821
Low       10570
High       2109
Name: count, dtype: int64

Activity features used:
- LoginAttempts_flag
- SensitiveFilesAccess_flag
- UnauthorizedSoftware_flag
- DataTransfer_flag
- NonWorkWebsitesVisited_flag
- PhysicalAccess_flag
- SocialEngineeringAttacks_flag
- PreviousIncidents_flag
- PublicInfoShared_flag
- InteractionWithMaliciousAccounts_flag
- InteractionWithMaliciousAccounts_malicious_flag
- BehaviorChange_flag
- NetworkInteraction_flag
- PoorInfoSecPractices_flag
- UploadSensitiveInformation_flag
- SendSensitiveInformation_flag
- AttemptedThumbDriveInsertion_flag
- SecurePrinting_flag

Total activity features: 18

Training model...
Model trained successfully!

MODEL EVALUATION

Accuracy: 100.00%

Classification Report:
                   precision    recall  f1-score   support

           Normal       1